# The Storyteller's Apprentice — Text Generation with Transformers

In [ ]:
import os, re, math, random, sys
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

In [ ]:
BATCH_SIZE = 64
MAX_LEN = 192          # length of x = [bos] + ids + [eos] + padding
T_LEN = MAX_LEN - 1    # 191 = length of inp / tgt
D_MODEL = 128
N_HEADS = 4
LR = 1e-3
EPOCHS = 20
PAD, BOS, EOS, UNK = 0, 1, 2, 3

## Task 1 — Open the Box

In [ ]:
def tokenize(s):
    return re.findall(r"[a-z']+|[0-9]+|[^\sa-z0-9]", s.lower())

In [ ]:
with open("data/TinyStories-valid.txt", encoding="utf-8") as f:
    text = f.read()
text = text.replace("\u2019", "'").replace("\u2018", "'")
stories = [s.strip() for s in text.split("<|endoftext|>")]
stories = [s for s in stories if s] # drop empty pieces
stories = [s for s in stories if len(tokenize(s)) <= 190] # whole stories only
train_stories = stories[0:8000]
val_stories = stories[8000:8500]
test_stories = stories[8500:9000]

In [ ]:
# counts for the report (recomputed without touching the fixed loading code above)
n_nonempty = sum(1 for s in text.split("<|endoftext|>") if s.strip())
print("Non-empty stories in file      :", n_nonempty)
print("Stories after 190-token filter :", len(stories))
print("Train / val / test sizes       :", len(train_stories), len(val_stories), len(test_stories))

In [ ]:
counts = Counter()
for s in train_stories:
    counts.update(tokenize(s))
kept = sorted([t for t, n in counts.items() if n >= 5], key=lambda t: (-counts[t], t))
itos = ["<pad>", "<bos>", "<eos>", "<unk>"] + kept
stoi = {t: i for i, t in enumerate(itos)}
V = len(itos)
assert len(stoi) == len(itos) and all(stoi[itos[k]] == k for k in range(V))
print("Vocabulary size V =", V)

In [ ]:
def encode(s):
    ids = [stoi.get(t, UNK) for t in tokenize(s)]
    assert len(ids) <= MAX_LEN - 2      # guaranteed by the 190-token filter
    x = [BOS] + ids + [EOS]
    return x + [PAD] * (MAX_LEN - len(x))

def build(ss):
    X = torch.tensor([encode(s) for s in ss], dtype=torch.long)
    return X[:, :-1].contiguous(), X[:, 1:].contiguous()

tr_inp, tr_tgt = build(train_stories)
va_inp, va_tgt = build(val_stories)
te_inp, te_tgt = build(test_stories)
print(tr_inp.shape, va_inp.shape, te_inp.shape)

In [ ]:
# 1.2  percentage of <unk> among word tokens of the encoded validation set (specials excluded)
def unk_percentage(ss):
    n_unk, n_tot = 0, 0
    for s in ss:
        ids = [stoi.get(t, UNK) for t in tokenize(s)]
        n_tot += len(ids)
        n_unk += sum(1 for i in ids if i == UNK)
    return 100.0 * n_unk / n_tot

print(f"<unk> in validation set: {unk_percentage(val_stories):.2f}%")

In [ ]:
# 1.3  story 0 of the training set
x0 = encode(train_stories[0])
print("First 15 tokens      :", tokenize(train_stories[0])[:15])
print("First 16 values of x :", x0[:16])
print("True values in padding mask:", int((tr_inp[0] != PAD).sum().item()))

In [ ]:
def make_loader(inp, tgt, shuffle):
    g = torch.Generator().manual_seed(SEED) if shuffle else None
    return torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(inp, tgt),
        batch_size=BATCH_SIZE, shuffle=shuffle, generator=g, drop_last=False)

# validation / test loaders (a fresh training loader is created inside every training run)
val_loader = make_loader(va_inp, va_tgt, False)
test_loader = make_loader(te_inp, te_tgt, False)
len(val_loader), len(test_loader)

## Task 2 — Learn to Look Back

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=128, n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.H, self.d = n_heads, d_model // n_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.o = nn.Linear(d_model, d_model)

    def forward(self, h, pad_mask, head_mask=None):
        B, T, D = h.shape
        split = lambda x: x.reshape(B, T, self.H, self.d).transpose(1, 2)   # (B, H, T, d)
        Q, K, Vv = split(self.q(h)), split(self.k(h)), split(self.v(h))
        S = Q @ K.transpose(-2, -1) / math.sqrt(self.d)                      # (B, H, T, T)
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=h.device))
        allowed = causal[None, None] & pad_mask.bool()[:, None, None, :]
        A = torch.softmax(S.masked_fill(~allowed, float("-inf")), dim=-1)
        O = A @ Vv                                                           # (B, H, T, d)
        if head_mask is not None:
            O = O * head_mask.to(device=O.device, dtype=O.dtype).reshape(1, self.H, 1, 1)
        O = O.transpose(1, 2).contiguous().reshape(B, T, D)
        return self.o(O), A

In [ ]:
# Checks
chk_inp, _ = next(iter(val_loader))          # first validation batch
B = chk_inp.shape[0]
pm = chk_inp != PAD

torch.manual_seed(SEED)
attn = CausalSelfAttention(D_MODEL, N_HEADS)
attn.eval()
h = torch.randn(B, T_LEN, D_MODEL)
with torch.no_grad():
    out1, A = attn(h, pm)

# 1. row sums over rows whose query position is a real token
row_sums = A.sum(-1)                                   # (B, H, T)
real_q = pm[:, None, :].expand_as(row_sums)
print(f"Row sums (real queries): min = {row_sums[real_q].min().item():.7f}, max = {row_sums[real_q].max().item():.7f}")

# 2. largest weight on a future position and on a padding key
future = torch.triu(torch.ones(T_LEN, T_LEN, dtype=torch.bool), diagonal=1)
max_future = A[:, :, future].max().item()
pad_key = (~pm)[:, None, None, :].expand_as(A)
max_pad = A[pad_key].max().item() if pad_key.any() else 0.0
print("Largest weight on a future position:", max_future)
print("Largest weight on a padding key    :", max_pad)

In [ ]:
# 3. tamper test
h2 = h.clone()
h2[:, 11:, :] = torch.randn(B, T_LEN - 11, D_MODEL)
with torch.no_grad():
    out2, _ = attn(h2, pm)
print("Max |difference| at positions 0..10:", (out1[:, :11] - out2[:, :11]).abs().max().item())

## Task 3 — Raise the Apprentice

In [ ]:
class Block(nn.Module):
    def __init__(self, D=128, H=4):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.att = CausalSelfAttention(D, H)
        self.drop = nn.Dropout(0.1)
        self.ffn = nn.Sequential(nn.Linear(D, 256), nn.GELU(), nn.Linear(256, D), nn.Dropout(0.1))

    def forward(self, h, pm, hm=None):
        a, A = self.att(self.ln1(h), pm, hm)
        h = h + self.drop(a)
        h = h + self.ffn(self.ln2(h))
        return h, A


class LM(nn.Module):
    def __init__(self, V, use_pos=True, D=128):
        super().__init__()
        self.use_pos = use_pos
        self.tok = nn.Embedding(V, D)
        if use_pos:
            self.pos = nn.Embedding(T_LEN, D)
        self.drop = nn.Dropout(0.1)
        self.blocks = nn.ModuleList([Block(D, N_HEADS) for _ in range(2)])
        self.lnf = nn.LayerNorm(D)
        self.out = nn.Linear(D, V)

    def forward(self, inp, head_masks=None):
        pm = inp != PAD
        h = self.tok(inp)
        if self.use_pos:
            h = h + self.pos(torch.arange(inp.shape[1], device=inp.device))[None]
        h = self.drop(h)
        As = []
        for i, blk in enumerate(self.blocks):
            h, A = blk(h, pm, None if head_masks is None else head_masks[i])
            As.append(A)
        return self.out(self.lnf(h)), As


def nparams(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
def evaluate(model, loader, head_masks=None):
    model.eval()
    device = next(model.parameters()).device
    tot, n, corr = 0.0, 0, 0
    with torch.no_grad():
        for inp, tgt in loader:
            inp, tgt = inp.to(device), tgt.to(device)
            logits, _ = model(inp, head_masks)
            fl, ft = logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1)
            tot += F.cross_entropy(fl, ft, ignore_index=PAD, reduction="sum").item()
            v = ft != PAD
            n += v.sum().item()
            corr += ((fl.argmax(-1) == ft) & v).sum().item()
    return tot / n, corr / n


def train_model(model):
    device = next(model.parameters()).device
    train_loader = make_loader(tr_inp, tr_tgt, True)     # fresh, seeded loader for every run
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    th, vh = [], []
    for ep in range(1, EPOCHS + 1):
        model.train()
        bl = []
        for inp, tgt in train_loader:
            inp, tgt = inp.to(device), tgt.to(device)
            optimizer.zero_grad()
            logits, _ = model(inp)
            loss = criterion(logits.reshape(-1, V), tgt.reshape(-1))
            loss.backward()
            optimizer.step()
            bl.append(loss.item())
        th.append(sum(bl) / len(bl))
        vl, _ = evaluate(model, val_loader)
        vh.append(vl)
        print(f"Epoch {ep:2d}: train {th[-1]:.4f}    val {vh[-1]:.4f}")
    return th, vh

In [ ]:
# 3.1 Build it
torch.manual_seed(SEED)
model = LM(V, use_pos=True).to(DEVICE)
print("Trainable parameters:", nparams(model))

In [ ]:
# 3.2 Train it
th, vh = train_model(model)
torch.save({"model": model.state_dict()}, "task3_model.pt")

In [ ]:
# 3.3 Plot it
ep = range(1, EPOCHS + 1)
plt.figure(figsize=(7, 4.5))
plt.plot(ep, th, marker="o", label="train loss")
plt.plot(ep, vh, marker="o", label="validation loss")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss")
plt.title("Task 3: training and validation loss")
plt.xticks(list(ep)); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 3.4 Test it
t_loss, t_acc = evaluate(model, test_loader)
print(f"Test loss {t_loss:.4f}   perplexity {math.exp(t_loss):.3f}   accuracy {100 * t_acc:.2f}%")

### Bonus — bigram baseline

In [ ]:
cp, cv = Counter(), Counter()
for ri, rt in zip(tr_inp.tolist(), tr_tgt.tolist()):
    for a, b in zip(ri, rt):
        if b != PAD:
            cp[(a, b)] += 1
            cv[a] += 1
nll, cnt = 0.0, 0
for ri, rt in zip(te_inp.tolist(), te_tgt.tolist()):
    for a, b in zip(ri, rt):
        if b != PAD:
            nll += -math.log((cp[(a, b)] + 1) / (cv[a] + V))
            cnt += 1
bigram_ppl = math.exp(nll / cnt)
print(pd.DataFrame({"model": ["bigram (add-one)", "transformer"],
                    "test perplexity": [round(bigram_ppl, 3), round(math.exp(t_loss), 3)]}).to_string(index=False))

## Task 4 — Detective Work: What Does It Remember?

In [ ]:
COMMON_MAX = 53   # IDs 4..53 are "common"

def callback_mask(inp, tgt):
    """cb[b,i]  : position i is a callback position
       match[b,i,j]: 1 <= j <= i and inp[j] == tgt[i]"""
    T = inp.shape[1]
    match = tgt[:, :, None] == inp[:, None, :]
    tri = torch.tril(torch.ones(T, T, dtype=torch.bool, device=inp.device))
    tri[:, 0] = False                       # j starts at 1 (skip <bos>)
    match = match & tri[None]
    cb = match.any(-1) & (tgt > COMMON_MAX)
    return cb, match


def analyse(model, loader, head_masks=None, want_scores=False):
    model.eval()
    device = next(model.parameters()).device
    nb = len(model.blocks)
    sc = torch.zeros(nb, N_HEADS, device=device)
    tot = cbt = oth = 0.0
    n = cbn = on = corr = 0
    with torch.no_grad():
        for inp, tgt in loader:
            inp, tgt = inp.to(device), tgt.to(device)
            logits, As = model(inp, head_masks)
            l = F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1),
                                ignore_index=PAD, reduction="none").reshape(tgt.shape)
            cb, match = callback_mask(inp, tgt)
            valid = tgt != PAD
            other = valid & ~cb
            tot += l[valid].sum().item(); n += valid.sum().item()
            cbt += l[cb].sum().item();    cbn += cb.sum().item()
            oth += l[other].sum().item(); on += other.sum().item()
            corr += ((logits.argmax(-1) == tgt) & valid).sum().item()
            if want_scores:
                mf = match[:, None].float()                  # (B,1,T,T)
                for b in range(nb):
                    row = (As[b] * mf).sum(-1)               # (B,H,T)
                    sc[b] += (row * cb[:, None, :].float()).sum((0, 2))
    return dict(loss=tot / n, acc=corr / n,
                cb_n=cbn, cb_loss=cbt / cbn if cbn else float("nan"),
                ot_n=on, ot_loss=oth / on if on else float("nan"),
                scores=(sc / cbn).cpu() if (want_scores and cbn) else None)

In [ ]:
# 4.1 Is it harder?
res_full = analyse(model, test_loader, want_scores=True)
print(pd.DataFrame({
    "position type": ["callback", "other"],
    "count": [res_full["cb_n"], res_full["ot_n"]],
    "mean loss": [round(res_full["cb_loss"], 4), round(res_full["ot_loss"], 4)],
}).to_string(index=False))

In [ ]:
# 4.2 Score the heads
scores = res_full["scores"].numpy()                      # (2, 4)
fig, ax = plt.subplots(figsize=(6, 3))
im = ax.imshow(scores, cmap="viridis")
for b in range(scores.shape[0]):
    for l in range(scores.shape[1]):
        ax.text(l, b, f"{scores[b, l]:.3f}", ha="center", va="center",
                color="white" if scores[b, l] < scores.max() * 0.6 else "black")
ax.set_xticks(range(N_HEADS)); ax.set_xticklabels([f"head {l + 1}" for l in range(N_HEADS)])
ax.set_yticks(range(2)); ax.set_yticklabels(["block 1", "block 2"])
ax.set_title("Callback score per head")
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

best_b, best_h = np.unravel_index(scores.argmax(), scores.shape)
low_b, low_h = np.unravel_index(scores.argmin(), scores.shape)
print(f"Best head  : block {best_b + 1}, head {best_h + 1} (score {scores[best_b, best_h]:.4f})")
print(f"Lowest head: block {low_b + 1}, head {low_h + 1} (score {scores[low_b, low_h]:.4f})")

In [ ]:
# 4.3 Watch it work
W = 25
cb_test, _ = callback_mask(te_inp, te_tgt)
story_idx = next(i for i in range(len(te_inp)) if cb_test[i, :W].sum().item() >= 2)
print("Using test story", story_idx)

model.eval()
with torch.no_grad():
    _, As_story = model(te_inp[story_idx:story_idx + 1].to(DEVICE))
Aw = As_story[best_b][0, best_h, :W, :W].cpu().numpy()

words = [itos[t] for t in te_inp[story_idx, :W].tolist()]
cb_pos = cb_test[story_idx, :W].tolist()
ylabels = [w + (" *" if c else "") for w, c in zip(words, cb_pos)]

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(Aw, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(W)); ax.set_xticklabels(words, rotation=90)
ax.set_yticks(range(W)); ax.set_yticklabels(ylabels)
ax.set_xlabel("key token"); ax.set_ylabel("query token  (* = callback position)")
ax.set_title(f"Block {best_b + 1}, head {best_h + 1} attention — test story {story_idx}, positions 0–24")
fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# 4.4 Head surgery
def masks_without(b_off=None, h_off=None):
    hm = [torch.ones(N_HEADS, device=DEVICE) for _ in range(len(model.blocks))]
    if b_off is not None:
        hm[b_off][h_off] = 0.0
    return hm

res_best = analyse(model, test_loader, head_masks=masks_without(best_b, best_h))
res_ctrl = analyse(model, test_loader, head_masks=masks_without(low_b, low_h))

surgery = pd.DataFrame({
    "setting": ["nothing removed",
                f"best head removed (block {best_b + 1}, head {best_h + 1})",
                f"control head removed (block {low_b + 1}, head {low_h + 1})"],
    "test loss": [res_full["loss"], res_best["loss"], res_ctrl["loss"]],
    "callback loss": [res_full["cb_loss"], res_best["cb_loss"], res_ctrl["cb_loss"]],
}).round(4)
print(surgery.to_string(index=False))

## Task 5 — The Story Thermostat

In [ ]:
PROMPT_LEN = 7

@torch.no_grad()
def generate(model, prompt, tau=None):
    model.eval()
    device = next(model.parameters()).device
    seq, ended = list(prompt), False
    while len(seq) < T_LEN:                       # stop when sequence reaches 191 tokens
        logits, _ = model(torch.tensor([seq], dtype=torch.long, device=device))
        nl = logits[0, -1, :].clone()
        nl[[PAD, BOS, UNK]] = -float("inf")
        if tau is None:
            nid = nl.argmax().item()
        else:
            nid = torch.multinomial(torch.softmax(nl / tau, -1), 1).item()
        seq.append(nid)
        if nid == EOS:
            ended = True
            break
    gen = seq[len(prompt):]
    if ended:
        gen = gen[:-1]                            # drop <eos>
    return gen, ended

In [ ]:
# 5.1 Generate
settings = [("greedy", None), ("temperature 0.7", 0.7), ("temperature 1.2", 1.2)]
results = {}
for name, tau in settings:
    torch.manual_seed(SEED)
    results[name] = [generate(model, te_inp[i, :PROMPT_LEN].tolist(), tau) for i in range(50)]
    print(f"{name}: generated {len(results[name])} stories")

In [ ]:
# 5.2 Compare
def distinct2(outs):
    pairs = [p for g, _ in outs for p in zip(g[:-1], g[1:])]
    return len(set(pairs)) / len(pairs) if pairs else float("nan")

rows = []
for name, _ in settings:
    outs = results[name]
    rows.append({"setting": name,
                 "mean length": np.mean([len(g) for g, _ in outs]),
                 "% ended with <eos>": 100 * np.mean([e for _, e in outs]),
                 "distinct-2": distinct2(outs)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# 5.3 Read them
dec = lambda ids: " ".join(itos[t] for t in ids)
print("PROMPT 0:", dec(te_inp[0, :PROMPT_LEN].tolist()))
print()
for name, _ in settings:
    print(f"[{name}]", dec(results[name][0][0]))
    print()

## Task 6 — The Blindfold Test

In [ ]:
# 6.1 Build and train the model without positional embeddings
torch.manual_seed(SEED)
no_pos_model = LM(V, use_pos=False).to(DEVICE)
print("Trainable parameters (no positions):", nparams(no_pos_model))
nth, nvh = train_model(no_pos_model)
torch.save({"model": no_pos_model.state_dict()}, "task6_model.pt")

In [ ]:
# 6.2 Compare the two models
res_pos = analyse(model, test_loader)
res_nopos = analyse(no_pos_model, test_loader)
cmp = pd.DataFrame({
    "model": ["with positions", "without positions"],
    "test loss": [res_pos["loss"], res_nopos["loss"]],
    "test perplexity": [math.exp(res_pos["loss"]), math.exp(res_nopos["loss"])],
    "test accuracy (%)": [100 * res_pos["acc"], 100 * res_nopos["acc"]],
    "callback loss": [res_pos["cb_loss"], res_nopos["cb_loss"]],
}).round(4)
print(cmp.to_string(index=False))

plt.figure(figsize=(7, 4.5))
plt.plot(ep, vh, marker="o", label="with positions")
plt.plot(ep, nvh, marker="o", label="without positions")
plt.xlabel("Epoch"); plt.ylabel("Validation loss")
plt.title("Task 6: validation loss with vs. without positional embeddings")
plt.xticks(list(ep)); plt.legend(); plt.grid(alpha=0.3)
plt.show()